# Visualize an output circuit

This notebook loads one of the generated QASM circuits and shows two views:

1. The complete Qiskit circuit diagram.
2. The qubit-interaction graph, where nodes are qubits and edges show two-qubit gates.

The default example is the `wisq_rules_hardware` output for `efficient_su2_10_r2`. Change `CIRCUIT_PATH` below to inspect another output.

In [ ]:
from collections import Counter
from pathlib import Path
import math

import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, qasm2
from qiskit.circuit.library import SXGate

# This path works whether Jupyter is started from the repository root or data/.
CIRCUIT_PATH = Path("data/artifacts/efficient_su2_10_r2/wisq_rules_hardware.qasm")
candidates = [CIRCUIT_PATH, Path("..") / CIRCUIT_PATH]
CIRCUIT_PATH = next((p for p in candidates if p.exists()), CIRCUIT_PATH)

if not CIRCUIT_PATH.exists():
    raise FileNotFoundError(f"Could not find circuit: {CIRCUIT_PATH.resolve()}")

custom_instructions = [qasm2.CustomInstruction("sx", 0, 1, constructor=SXGate, builtin=True)]
circuit = qasm2.load(str(CIRCUIT_PATH), custom_instructions=custom_instructions)
print(f"Loaded: {CIRCUIT_PATH.resolve()}")
print(f"Qubits: {circuit.num_qubits}")
print(f"Classical bits: {circuit.num_clbits}")
print(f"Gates: {circuit.size()}")
print(f"Depth: {circuit.depth()}")

## Circuit diagram

In [ ]:
# For large circuits, increase or decrease fold to control line wrapping.
circuit.draw(output="mpl", fold=-1, scale=0.7)

## Qubit-interaction graph

Each node is a logical qubit. An edge means that the two qubits participate in at least one two-qubit gate; thicker edges indicate more interactions. This describes the circuit's logical interaction shape, not necessarily the hardware chip's physical coupling map.

In [ ]:
interaction_counts = Counter()
for instruction in circuit.data:
    qubits = tuple(circuit.find_bit(q).index for q in instruction.qubits)
    if len(qubits) == 2:
        edge = tuple(sorted(qubits))
        interaction_counts[edge] += 1

fig, ax = plt.subplots(figsize=(10, 7))
n = circuit.num_qubits
radius = max(2.5, n / 3.5)
positions = {
    q: (radius * math.cos(2 * math.pi * q / n),
        radius * math.sin(2 * math.pi * q / n))
    for q in range(n)
}

for (q1, q2), count in interaction_counts.items():
    x1, y1 = positions[q1]
    x2, y2 = positions[q2]
    ax.plot([x1, x2], [y1, y2], color="steelblue",
            linewidth=1.5 + 1.5 * count / max(interaction_counts.values()),
            alpha=0.7, zorder=1)
    ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(count),
            color="darkblue", fontsize=9, ha="center", va="center",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.75), zorder=3)

xs = [positions[q][0] for q in range(n)]
ys = [positions[q][1] for q in range(n)]
ax.scatter(xs, ys, s=850, color="darkorange", edgecolor="black", zorder=2)
for q, (x, y) in positions.items():
    ax.text(x, y, f"q[{q}]", ha="center", va="center",
            fontsize=10, fontweight="bold", zorder=4)

ax.set_title("Logical qubit interaction graph")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

print(f"Two-qubit interaction edges: {len(interaction_counts)}")
print("Edges and interaction counts:")
for edge, count in sorted(interaction_counts.items()):
    print(f"  q[{edge[0]}] -- q[{edge[1]}]: {count}")